In [ ]:
import sys
from pathlib import Path
import os
import time
import pickle
import pandas as pd
from tqdm.auto import tqdm
import asyncio

# Добавляем backend в PYTHONPATH
BACKEND_DIR = Path.cwd().parent / 'backend'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

# Импорты из проекта
from config import OPENAI_API_KEY, OPENAI_BASE_URL, TEMPERATURE, MAX_TOKENS
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Функция извлечения ответа из выдачи решателя
from agents.solver import extract_answer

/home/tas/.cache/pypoetry/virtualenvs/ai-mas-hse-project-VtkJAIkS-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Вспомогательная функция извлечения токенов (для информации, если нужно)
def get_token_usage(response):
    if hasattr(response, 'usage_metadata') and response.usage_metadata:
        inp = response.usage_metadata.get("input_tokens", 0)
        out = response.usage_metadata.get("output_tokens", 0)
        return inp, out
    if hasattr(response, 'response_metadata'):
        token_usage = response.response_metadata.get("token_usage", {})
        inp = token_usage.get("prompt_tokens", 0)
        out = token_usage.get("completion_tokens", 0)
        return inp, out
    return 0, 0


def load_standard_dataset():
    """Загружает стандартный датасет задач."""
    with open(STANDARD_DATASET_PATH, 'rb') as f:
        data = pickle.load(f)
    df = pd.DataFrame(data)
    df = df.rename(columns={'answer': 'ground_truth'})
    return df


def load_mcp_dataset():
    """Загружает датасет MCP-уравнений."""
    with open(MCP_DATASET_PATH, 'rb') as f:
        data = pickle.load(f)
    df = pd.DataFrame(data)
    df = df.rename(columns={'answer': 'ground_truth'})  # на всякий случай
    return df

# 1. Загрузка датасетов

In [ ]:
DATASET_DIR = Path.cwd().parent / 'static_dataset'
STANDARD_DATASET_PATH = DATASET_DIR / 'list_dict_with_tasks_update.pkl'
MCP_DATASET_PATH = DATASET_DIR / 'mcp_equations.pkl'

standard_df = load_standard_dataset()
mcp_df = load_mcp_dataset()

print(f"Стандартный датасет: {len(standard_df)} задач")
print(f"MCP-датасет: {len(mcp_df)} задач")
standard_df.head(2)

Стандартный датасет: 147 задач
MCP-датасет: 110 задач


,id,topic,subtopic,complexity_level,problem,solution,ground_truth,complexity_level_text
0,30262,Алгебра и арифметика,Арифметика. Устный счет и т.п,2,"Из книги выпал кусок, первая страница которого...",,496 страниц (248 листов),Легкий
1,30289,Алгебра и арифметика,Четность и нечетность,2,Можно ли доску размером 5×5 заполнить доминошк...,"Общее количество клеток (25) не делится на 2, ...",Нельзя,Легкий


In [ ]:
mcp_df.head()

,id,problem,ground_truth,hint
0,1,Решите квадратное уравнение: $x^2 - 5x + 6 = 0...,"2, 3",Квадратное уравнение
1,2,Решите квадратное уравнение: $x^2 + 3x + 2 = 0...,"-2, -1",Квадратное уравнение
2,3,Решите квадратное уравнение: $2x^2 - 7x + 3 = ...,"0.5, 3",Квадратное уравнение
3,4,Решите квадратное уравнение: $x^2 - 4x + 4 = 0...,"2, 2",Квадратное уравнение
4,5,Решите квадратное уравнение: $3x^2 + 5x - 2 = ...,"-2, 0.3333333333333333",Квадратное уравнение


# 2. LLM-судья для сравнения ответов

In [ ]:
# Модель для судьи
JUDGE_MODEL_ID = "deepseek/deepseek-v3.2"  # или "gpt-4o"

In [ ]:
def create_judge_llm():
    """Создаёт LLM для судьи (не зависит от модели решателя)."""
    return ChatOpenAI(
        model=JUDGE_MODEL_ID,
        temperature=0.0,
        max_tokens=3000,
        timeout=180,
        max_retries=3,
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
    )

JUDGE_PROMPT = """
Ты математический эксперт. Сравни ответ решателя с правильным ответом и определи, эквивалентны ли они математически.

Ответ решателя (может содержать лишний текст):
{solver_answer}

Правильный ответ:
{ground_truth}

Правила:
- Извлеки финальный ответ из текста решателя (игнорируй объяснения, пометки «Ответ:», «x =»).
- Сравни математическую суть: упрости выражения, приведи к общему виду.
- Для уравнений/неравенств сравни множества корней/решений (порядок не важен).
- Числа сравнивай с учетом возможного округления (например, 3.1416 ≈ π, 1/3 ≈ 0.333).
- Если ответы математически идентичны – YES, иначе NO.

Выведи СТРОГО одно слово: YES или NO. Никаких пояснений.
"""

In [ ]:
def llm_judge(solver_answer: str, ground_truth: str, judge_llm: ChatOpenAI) -> bool:
    """Возвращает True, если судья считает ответы эквивалентными."""
    if solver_answer.strip() == ground_truth.strip():
        return True

    prompt = JUDGE_PROMPT.format(solver_answer=solver_answer, ground_truth=ground_truth)
    messages = [HumanMessage(content=prompt)]
    try:
        response = judge_llm.invoke(messages)
        verdict = response.content.strip().upper()
        return verdict == "YES"
    except Exception as e:
        print(f"Ошибка при вызове судьи: {e}")
        return False

# 3. Функции решателей для двух типов задач

## 3.1. Обычный решатель (LLM + промт)

In [ ]:
def solve_standard(problem: str, llm: ChatOpenAI, system_prompt: str) -> str:
    """Решает обычную задачу с помощью LLM, возвращает извлечённый ответ."""
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Реши задачу: {problem}")
    ]
    try:
        response = llm.invoke(messages)
        full_text = response.content
        answer = extract_answer(full_text)
        return answer
    except Exception as e:
        print(f"Ошибка при решении (standard): {e}")
        return ""

## 3.2. MCP-решатель (через математический сервер)

In [ ]:
import config as cfg
from agents.mcp_solver import MCPClient, MCPSolverAgent, MCPSolverAgentSync

def solve_mcp(problem: str, model_id: str) -> str:
    """
    Решает MCP-задачу, используя указанную модель.
    Возвращает финальный ответ.
    """
    cfg.MODEL_ID = model_id
    cfg._llm = None   # сброс кэша

    MCPClient._instance = None
    MCPClient._initialized = False

    server_path = str(Path.cwd().parent / 'calculator_server.py')
    client = MCPClient(server_path=server_path)
    mcp_solver = MCPSolverAgentSync()
    try:
        result = mcp_solver.solve(problem)
        answer = result.get('answer', '')
        return answer
    except Exception as e:
        print(f"Ошибка при решении (MCP): {e}")
        return ""
    finally:
        mcp_solver.close()

## 3.3. Универсальная обёртка для выбора решателя по типу датасета

In [ ]:
def get_answer(dataset_type: str, problem: str, llm_or_model, system_prompt: str = None) -> str:
    """
    dataset_type: 'standard' или 'mcp'
    llm_or_model: для standard – экземпляр ChatOpenAI,
                  для mcp – строка model_id
    """
    if dataset_type == 'standard':
        return solve_standard(problem, llm_or_model, system_prompt)
    else:  # mcp
        return solve_mcp(problem, llm_or_model)

# 4. Расчёт Success Rate для одного цикла (модель + промт)

In [ ]:
def evaluate_dataset(
    dataset_df: pd.DataFrame,
    dataset_type: str,
    llm_or_model,
    system_prompt: str,
    judge_llm: ChatOpenAI,
    limit: int = None
) -> dict:
    """
    Прогоняет датасет через решатель и судью.
    Возвращает словарь со статистикой.
    """
    df = dataset_df.copy()
    if limit:
        df = df.head(limit)

    problems = df['problem'].tolist()
    ground_truths = df['ground_truth'].tolist()
    total = len(problems)
    correct = 0

    results = []
    for prob, gt in tqdm(zip(problems, ground_truths), total=total, desc=f"{dataset_type}"):
        try:
            answer = get_answer(dataset_type, prob, llm_or_model, system_prompt)
            is_correct = llm_judge(answer, gt, judge_llm)
            if is_correct:
                correct += 1
            results.append({
                'problem': prob,
                'ground_truth': gt,
                'solver_answer': answer,
                'correct': is_correct
            })
        except Exception as e:
            results.append({
                'problem': prob,
                'ground_truth': gt,
                'solver_answer': 'ERROR',
                'correct': False
            })

    success_rate = correct / total if total > 0 else 0.0
    return {
        'total': total,
        'correct': correct,
        'success_rate': success_rate,
        'details': results
    }

# 5. Параметры эксперимента: модели и промпты

In [ ]:
PROMPTS = {
    "cot": (
        "# Роль\n"
        "Ты – опытный математик-педагог.\n\n"
        "# Контекст\n"
        "Тебе дана математическая задача. Твоя цель – решить её и предоставить ответ.\n\n"
        "# CoT\n"
        "Используй метод Chain of Thought: пошагово рассуждай, описывая каждый этап решения.\n\n"
        "# Формат вывода\n"
        "Строго соблюдай следующий формат:\n"
        "РАССУЖДЕНИЕ: [подробное пошаговое размышление]\n"
        "РЕШЕНИЕ: [краткая запись решения]\n"
        "ОТВЕТ: [итоговый числовой ответ или формула]\n\n"
        "В поле ОТВЕТ помести только сам ответ, без дополнительных слов."
    ),
    "no_cot": (
        "# Роль\n"
        "Ты – лаконичный решатель математических задач.\n\n"
        "# Контекст\n"
        "Тебе дана математическая задача. Твоя цель – решить её и предоставить ответ.\n\n"
        "# CoT\n"
        "Не используй развёрнутые рассуждения. Дай решение сразу, без цепочки мыслей.\n\n"
        "# Формат вывода\n"
        "Строго соблюдай следующий формат:\n"
        "РЕШЕНИЕ: [краткое решение]\n"
        "ОТВЕТ: [итоговый числовой ответ или формула]\n\n"
        "Не добавляй ничего сверх этого. В поле ОТВЕТ – только ответ."
    ),
    "vanilla": "Реши задачу и напиши ответ."
}


# 6. Цикл по всем комбинациям модель × промпт × датасет

In [ ]:
# Список моделей для тестирования (можно менять)
MODELS_TO_TEST = [
    "deepseek/deepseek-v3.2",
    "qwen/qwen3.5-397b-a17b",
    "openai/gpt-5.2",
    "anthropic/claude-opus-4.5"
]

In [ ]:
# Судья – одна на все эксперименты
judge_llm = create_judge_llm()

In [ ]:
MAX_TASKS = None  # поставьте, например, 50 для пробного прогона. Если хочешь прогнать все, то напиши None

all_metrics = []

In [ ]:
for model_id in MODELS_TO_TEST:
    # Для стандартных задач создаём LLM с текущей моделью
    llm_standard = ChatOpenAI(
        model=model_id,
        temperature=0.7,
        max_tokens=3000, #int(MAX_TOKENS) if MAX_TOKENS else None,
        timeout=180,
        max_retries=3,
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
    )

    for prompt_name, prompt_text in PROMPTS.items():
        # 1. Стандартный датасет
        print(f"\n--- Модель: {model_id}, Промпт: {prompt_name}, Датасет: standard ---")
        standard_stats = evaluate_dataset(
            standard_df,
            'standard',
            llm_standard,
            prompt_text,
            judge_llm,
            limit=MAX_TASKS
        )
        all_metrics.append({
            'model': model_id,
            'prompt': prompt_name,
            'dataset': 'standard',
            'total': standard_stats['total'],
            'correct': standard_stats['correct'],
            'success_rate': standard_stats['success_rate']
        })

        # 2. MCP-датасет (модель задаётся строкой)
        print(f"--- Модель: {model_id}, Промпт: {prompt_name}, Датасет: mcp ---")
        mcp_stats = evaluate_dataset(
            mcp_df,
            'mcp',
            model_id,        # для MCP передаём model_id
            prompt_text,     # промпт здесь не используется, но сохраним для отчётности
            judge_llm,
            limit=MAX_TASKS
        )
        all_metrics.append({
            'model': model_id,
            'prompt': prompt_name,
            'dataset': 'mcp',
            'total': mcp_stats['total'],
            'correct': mcp_stats['correct'],
            'success_rate': mcp_stats['success_rate']
        })

# 7. Сводная таблица результатов

In [ ]:
results_df = pd.DataFrame(all_metrics)
results_df['success_rate_percent'] = results_df['success_rate'] * 100
results_df.drop(columns=['success_rate'])


# Вычисляем Combined SR как (сумма correct) / (сумма total) для каждой группы (model, prompt)
def compute_combined(group):
    total_correct = group['correct'].sum()
    total_total = group['total'].sum()
    combined = (total_correct / total_total) * 100
    return combined.round(2)

combined_sr = results_df.groupby(['model', 'prompt']).apply(compute_combined).reset_index(name='combined_sr')

# Добавляем колонку в исходный df для удобства
results_df = results_df.merge(combined_sr, on=['model', 'prompt'])

results_df

,model,prompt,dataset,total,correct,success_rate_percent,combined_sr
0,deepseek/deepseek-v3.2,cot,standard,147,134,91.15,88.72
1,deepseek/deepseek-v3.2,cot,mcp,110,94,85.45,88.72
2,deepseek/deepseek-v3.2,no_cot,standard,147,124,84.35,85.21
3,deepseek/deepseek-v3.2,no_cot,mcp,110,95,86.36,85.21
4,deepseek/deepseek-v3.2,vanilla,standard,147,117,79.59,80.93
5,deepseek/deepseek-v3.2,vanilla,mcp,110,91,82.72,80.93
6,qwen/qwen3.5-397b,cot,standard,147,130,88.43,86.38
7,qwen/qwen3.5-397b,cot,mcp,110,92,83.63,86.38
8,qwen/qwen3.5-397b,no_cot,standard,147,123,83.67,83.66
9,qwen/qwen3.5-397b,no_cot,mcp,110,92,83.63,83.66


In [ ]:
results_df.to_parquet('final_metrics_for_vkr.parquet')

# 8. Влияние MCP-агента

In [ ]:
results_df_mcp = pd.DataFrame(all_metrics)
results_df_mcp['success_rate_percent'] = results_df_mcp['success_rate'] * 100
results_df_mcp.drop(columns=['success_rate'])

# Каждая вторая строчка - это качество решения без MCP-агента, а только за счет модельной генерации
results_df_mcp

,model,prompt,dataset,total,correct,success_rate_percent
0,DeepSeek V3.2,CoT,mcp,110,94,85.45
1,DeepSeek V3.2,CoT,mcp,110,81,73.63
2,Qwen3.5 397B,CoT,mcp,110,92,83.63
3,Qwen3.5 397B,CoT,mcp,110,76,69.09
4,GPT-5.2,CoT,mcp,110,96,87.27
5,GPT-5.2,CoT,mcp,110,85,77.27
6,Claude Opus 4.5,CoT,mcp,110,96,87.27
7,Claude Opus 4.5,CoT,mcp,110,87,79.09


In [ ]:
results_df_mcp.to_parquet('mcp_influence_for_vkr.parquet')